# SCARLET: Semantic Clinical Alignment for Radiology Language Evaluation and Training

## Motivation

Radiology report generation aims to automatically generate clinical reports from chest X-ray images using vision-language models.

Although recent encoder-decoder models can generate fluent medical text, fluency alone is not enough in clinical settings. A generated report must also preserve the correct clinical meaning.

One major challenge is semantic correctness. Two reports may describe the same finding using different wording, while traditional lexical metrics such as BLEU may still assign low scores.

For example:

| Ground Truth | Prediction | BLEU Behavior |
|---|---|---|
| enlarged heart | cardiomegaly observed | low overlap |
| pleural fluid | pleural effusion | partial overlap |

Another challenge is hallucination, where the model generates unsupported or clinically incorrect findings. These errors are especially important in medical applications because fluent reports may still contain misleading information.

In addition, radiology datasets are often highly imbalanced, with normal studies appearing much more frequently than rare abnormalities. This can bias models toward generic or repetitive report generation.

To address these challenges, the proposed SCARLET framework focuses on:
- semantic-aware learning
- clinically-aware evaluation
- hallucination analysis
- imbalance-aware training
- semantic-clinical alignment

# 1. Environment Setup

This section initializes the experimental environment used throughout the notebook.

To improve reproducibility and training stability, we:
- import all required libraries
- fix random seeds
- configure GPU usage
- check package versions

Reproducibility is especially important in deep learning experiments because small differences in initialization or hardware can affect model behavior and evaluation results.

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score

from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

from tqdm import tqdm

warnings.filterwarnings("ignore")

In [ ]:
# setting the seed for reproducibility
SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed set to: {SEED}")

In [ ]:
# gpu setup

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():

    print("GPU Name:", torch.cuda.get_device_name(0))

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

# 2. Dataset Preparation

This project uses the MIMIC-CXR dataset, which is one of the largest publicly available chest X-ray datasets for medical vision-language research.

MIMIC-CXR contains:
- chest X-ray images
- paired radiology reports
- study-level metadata
- multiple views per study

The dataset is widely used in radiology report generation research because it provides realistic clinical reporting data at large scale.

In this section, we:
- load the dataset
- organize image-report pairs
- perform UID-level splitting
- preprocess chest X-ray images
- normalize inputs for model training

## 2.1 Downloading MIMIC-CXR in Google Colab

The dataset is downloaded directly from PhysioNet using authenticated access.

Since the dataset is large, downloading can take significant time depending on:
- internet speed
- Colab runtime
- storage availability

It is recommended to use:
- Google Drive mounting
- persistent storage
- high-RAM runtime when possible

In [ ]:
# from google.colab import drive

# drive.mount('/content/drive')

# DATASET_DIR = "/content/drive/MyDrive/mimic_cxr"

# os.makedirs(DATASET_DIR, exist_ok=True)

# print("Dataset directory ready.")

In [ ]:
!wget -r -N -c -np \
--user YOUR_USERNAME \
--ask-password \
https://physionet.org/files/mimic-cxr-jpg/2.1.0/ \
-P /content/drive/MyDrive/mimic_cxr

## 2.2 Dataset Structure

The MIMIC-CXR dataset contains:
- JPG chest X-ray images
- radiology reports
- metadata CSV files

The main metadata files include:
- study information
- image paths
- report associations
- patient identifiers

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/mimic_cxr"

print(DATA_ROOT)

metadata_path = os.path.join(
    DATA_ROOT,
    "physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv.gz"
)

split_path = os.path.join(
    DATA_ROOT,
    "physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv.gz"
)

metadata_df = pd.read_csv(metadata_path)

split_df = pd.read_csv(split_path)

print("Metadata Shape:", metadata_df.shape)

print("Split Shape:", split_df.shape)

## 2.3 UID-Level Splitting

One important issue in medical AI is data leakage.

If images from the same patient appear in both training and testing sets, the model may memorize patient-specific patterns instead of learning generalizable medical features.

To avoid this, splitting is performed using patient or study identifiers rather than random image-level splitting.

This improves:
- fairness
- reproducibility
- evaluation reliability

In [ ]:
metadata_df[[
    "subject_id",
    "study_id",
    "dicom_id"
]].head()